# View a Simulated Patient Heart OBJ

This notebook provides a simple interface to load and view a **simulated patient** OBJ file (generated by scaling components of the segmented healthy model).

**Expected location:** `../heart_models/simulated_patient_models/*.obj`

You can either:
- set `OBJ_PATH` directly, or
- pick a filename from the discovered list.

In [6]:
# Imports + Plotly renderer setup
import os
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

# Prefer VS Code renderer; fall back if needed
if 'vscode' in pio.renderers:
    pio.renderers.default = 'vscode'
elif 'plotly_mimetype' in pio.renderers:
    pio.renderers.default = 'plotly_mimetype'
else:
    pio.renderers.default = 'notebook'

print('Plotly renderer:', pio.renderers.default)


Plotly renderer: vscode


In [7]:
# Locate simulated-patient OBJ files (robust to notebook location)
from pathlib import Path

# Heuristic: walk upward from the current working directory until we find a marker
# that identifies the repo/project root.
# Markers chosen to match this project layout.
PROJECT_MARKERS = [
    Path('heart_models') / 'healthy.obj',
    Path('heart_models') / 'simulated_patient_models',
]

def find_project_root(start: Path, markers=PROJECT_MARKERS, max_up: int = 8) -> Path:
    start = start.resolve()
    cur = start
    for _ in range(max_up + 1):
        if any((cur / m).exists() for m in markers):
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError(
        "Could not infer PROJECT_ROOT by searching upward from: "
        f"{start}. Looked for markers: {markers}"
    )

# Use CWD as the anchor because it reflects where the kernel is running,
# regardless of where this .ipynb file was moved.
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
SIM_DIR = PROJECT_ROOT / 'heart_models' / 'simulated_patient_models'

print('CWD:', PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)
print('Simulated OBJ dir:', SIM_DIR)
print('Exists:', SIM_DIR.exists())

obj_files = sorted(SIM_DIR.glob('*.obj')) if SIM_DIR.exists() else []
print('Found simulated OBJ files:', len(obj_files))

# Show a few candidates
for p in obj_files[:20]:
    print('-', p.name)


CWD: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/notebooks
Project root: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026
Simulated OBJ dir: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_patient_models
Exists: True
Found simulated OBJ files: 59
- simulated_patient0.obj
- simulated_patient1.obj
- simulated_patient10.obj
- simulated_patient11.obj
- simulated_patient12.obj
- simulated_patient13.obj
- simulated_patient14.obj
- simulated_patient15.obj
- simulated_patient16.obj
- simulated_patient17.obj
- simulated_patient18.obj
- simulated_patient19.obj
- simulated_patient2.obj
- simulated_patient20.obj
- simulated_patient21.obj
- simulated_patient22.obj
- simulated_patient23.obj
- simulated_patient24.obj
- simulated_patient25.obj
- simulated_patient26.obj


In [8]:
# Select which OBJ(s) to view (patient 1 by default)
PAT_ID = 1

# Simulated model
SIM_OBJ_PATH = SIM_DIR / f'simulated_patient{PAT_ID}.obj'

# Actual model (from patient_models)
PATIENT_MODEL_DIR = PROJECT_ROOT / 'heart_models' / 'patient_models'
ACTUAL_OBJ_PATH = PATIENT_MODEL_DIR / f'heart_model_pat{PAT_ID}.obj'

print('Selected simulated:', SIM_OBJ_PATH)
print('Selected actual:', ACTUAL_OBJ_PATH)


Selected simulated: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_patient_models/simulated_patient1.obj
Selected actual: /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/patient_models/heart_model_pat1.obj


In [9]:
# OBJ loader (vertices only) + optional downsampling

def load_obj_vertices(path: Path) -> np.ndarray:
    verts = []
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('v '):
                parts = line.strip().split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return np.asarray(verts, dtype=np.float32)


def downsample_points(xyz: np.ndarray, max_points: int = 120_000, seed: int = 0) -> np.ndarray:
    if max_points is None or xyz.shape[0] <= max_points:
        return xyz
    rng = np.random.default_rng(seed)
    idx = rng.choice(xyz.shape[0], size=max_points, replace=False)
    return xyz[idx]


assert SIM_OBJ_PATH is not None and SIM_OBJ_PATH.exists(), (
    'SIM_OBJ_PATH is not set or does not exist. '
    'Set SIM_OBJ_PATH to a valid file in simulated_patient_models.'
)
assert ACTUAL_OBJ_PATH is not None and ACTUAL_OBJ_PATH.exists(), (
    'ACTUAL_OBJ_PATH is not set or does not exist. '
    'Set ACTUAL_OBJ_PATH to a valid file in patient_models.'
)

sim_verts = load_obj_vertices(SIM_OBJ_PATH)
actual_verts = load_obj_vertices(ACTUAL_OBJ_PATH)

print('Loaded simulated verts:', sim_verts.shape)
print('Loaded actual verts:', actual_verts.shape)


Loaded simulated verts: (79686, 3)
Loaded actual verts: (172762, 3)


In [10]:
# Render simulated and actual patient OBJs in separate plots

MAX_POINTS = 150_000  # reduce if rendering is slow
POINT_SIZE_SIM = 1.6
POINT_SIZE_ACTUAL = 1.6

COLOR_SIM = 'rgb(220,40,40)'
COLOR_ACTUAL = 'rgb(80,160,255)'

sim_pts = downsample_points(sim_verts, max_points=MAX_POINTS, seed=0)
actual_pts = downsample_points(actual_verts, max_points=MAX_POINTS, seed=1)

# Center each for nicer viewing (separately)
sim_pts = sim_pts - sim_pts.mean(axis=0)
actual_pts = actual_pts - actual_pts.mean(axis=0)

# Simulated
fig_sim = go.Figure(
    data=[
        go.Scatter3d(
            x=sim_pts[:, 0], y=sim_pts[:, 1], z=sim_pts[:, 2],
            mode='markers',
            name=f'Simulated {SIM_OBJ_PATH.name}',
            marker=dict(size=POINT_SIZE_SIM, color=COLOR_SIM, opacity=0.9),
        )
    ]
)
fig_sim.update_layout(
    title=f'Patient {PAT_ID}: Simulated ({SIM_OBJ_PATH.name})',
    scene=dict(aspectmode='data', xaxis=dict(color='#666'), yaxis=dict(color='#666'), zaxis=dict(color='#666')),
    margin=dict(l=0, r=0, b=0, t=60),
    height=760,
    paper_bgcolor='white',
)

# Actual
fig_act = go.Figure(
    data=[
        go.Scatter3d(
            x=actual_pts[:, 0], y=actual_pts[:, 1], z=actual_pts[:, 2],
            mode='markers',
            name=f'Actual {ACTUAL_OBJ_PATH.name}',
            marker=dict(size=POINT_SIZE_ACTUAL, color=COLOR_ACTUAL, opacity=0.9),
        )
    ]
)
fig_act.update_layout(
    title=f'Patient {PAT_ID}: Actual ({ACTUAL_OBJ_PATH.name})',
    scene=dict(aspectmode='data', xaxis=dict(color='#666'), yaxis=dict(color='#666'), zaxis=dict(color='#666')),
    margin=dict(l=0, r=0, b=0, t=60),
    height=760,
    paper_bgcolor='white',
)

fig_sim.show()
fig_act.show()
